# Hyperparameter tuning

## Objective

The objective of this notebook is to optimize the five models selected
during model comparison and identify the strongest candidate using
cross-validation performance and stability.

Hyperparameters are optimized primarily using the F1-score for class 1
(churn), balancing precision and recall while avoiding the use of the
held-out test set for model selection.

In [21]:
import pandas as pd
from pathlib import Path
from IPython.display import display
import joblib

# Validation
from sklearn.utils.validation import check_is_fitted

# Models
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.pipeline import Pipeline

## Load data split, artifact preprocessor and top 5 models

The train/test split generated in the preprocessing stage is loaded
without creating a new split. This preserves consistency across the
modeling notebooks.

The preprocessing artifact is reused so all candidate models receive
the same feature transformations established previously.

The five candidate models were selected in the previous model
comparison notebook.

In [2]:
SPLIT_DATA_DIR = Path("../data/processed/splits")

train_data = pd.read_csv(SPLIT_DATA_DIR / "train.csv")
test_data = pd.read_csv(SPLIT_DATA_DIR / "test.csv")

X_train = train_data.drop(columns="Churn")
y_train = train_data["Churn"]

X_test = test_data.drop(columns="Churn")
y_test = test_data["Churn"]

In [3]:
# Row consistency
assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)

# Expected feature structure
assert X_train.shape[1] == X_test.shape[1]
assert X_train.columns.equals(X_test.columns)

# Target should not be present in X
assert "Churn" not in X_train.columns
assert "Churn" not in X_test.columns

# Binary target validation
assert set(y_train.unique()).issubset({0, 1})
assert set(y_test.unique()).issubset({0, 1})

# No missing target values
assert y_train.notna().all()
assert y_test.notna().all()

print("Split data validated successfully.")

Split data validated successfully.


### Load data split validation
Before hyperparameter tuning, structural checks are performed to ensure:

- X and y contain the same number of observations.
- Train and test have identical feature structures.
- The target variable is excluded from X.
- The target remains binary.
- No missing target values are present.

In [4]:
MODELS_DIR = Path("../artifacts")
PREPROCESSOR_PATH = MODELS_DIR / "preprocessor.joblib"
if not PREPROCESSOR_PATH.exists():
    raise FileNotFoundError(f"preprocessor not found at {PREPROCESSOR_PATH.resolve()}")
preprocessor = joblib.load(PREPROCESSOR_PATH)

In [5]:
# Confirm that the preprocessing artifact is fitted
check_is_fitted(preprocessor)

# Confirm expected input feature count
assert hasattr(preprocessor, "feature_names_in_")
assert len(preprocessor.feature_names_in_) == X_train.shape[1]

# Confirm training columns match preprocessor input schema
assert set(preprocessor.feature_names_in_) == set(X_train.columns)

print("Preprocessor validated successfully.")

Preprocessor validated successfully.


In [6]:
TOP_MODEL_PATH = Path("../data/processed/top_5_models.csv")

top_5_models = pd.read_csv(TOP_MODEL_PATH)

## Select models for tuning

These five models were selected in the previous model-comparison
stage based on their cross-validation performance and variability.

Only these finalists advance to hyperparameter optimization.

In [7]:
top_5_models

,model
0,gaussian_naive_bayes
1,logistic_regression
2,gradient_boosting
3,support_vector_machine
4,lightgbm


## Build candidate pipelines

Each estimator is wrapped inside a Pipeline together with the
preprocessing stage.

This ensures preprocessing is fitted independently inside each
cross-validation training fold, reducing the risk of data leakage
and ensuring that hyperparameter evaluation includes the complete
ML workflow.

In [8]:
models = {
    "logistic_regression": LogisticRegression(),
    "gradient_boosting": GradientBoostingClassifier(),
    "support_vector_machine": SVC(),
    "gaussian_naive_bayes": GaussianNB(),
    "lightgbm": LGBMClassifier(verbose=0)
}

In [9]:
models_pipeline = {
    model_name: Pipeline([('preprocessor', preprocessor), ('model', estimator)])
    for model_name, estimator in models.items()
}

## Define candidates hyperparameter search spaces

A model-specific hyperparameter search space is defined for each
candidate.

The search spaces focus on hyperparameters with a meaningful impact
on model complexity, regularization, class imbalance or boosting
behavior.

In [10]:
parameter_grids = {
    "logistic_regression":{
        "search_type": "random",
        "params": {
            "model__C": [0.01,0.5,1.0,1.5,2.0],
            "model__dual": [True, False],
            "model__class_weight": [None, "balanced"],
            "model__solver": ["liblinear", "lbfgs","newton-cholesky","sag","saga"]
        }
    },
    "gradient_boosting": {
        "search_type": "random",
        "params": {
            "model__n_estimators": [50, 100, 150, 200],
            "model__learning_rate": [0.01, 0.05, 0.1, 0.2],
            "model__max_depth": [2, 3, 4],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__subsample": [0.7, 0.85, 1.0]
        }
    },

    "support_vector_machine": {
        "search_type": "random",
        "params":{
            "model__C": [0.1, 0.5, 1, 2, 10],
            "model__gamma": ["scale", "auto", 0.001, 0.01, 0.1],
            "model__kernel": ["rbf"],
            "model__class_weight": [None, "balanced"]
        }
    },

    "gaussian_naive_bayes": {
        "search_type": "grid",
        "params":{
            "model__var_smoothing":[1e-7, 1e-8, 1e-9, 1e-10, 1e-11]
        }
    },

    "lightgbm": {
        "search_type": "random",
        "params":{
            "model__n_estimators": [100, 200, 300, 500],
            "model__learning_rate": [0.01, 0.05, 0.1],
            "model__num_leaves": [15, 31, 63],
            "model__max_depth": [-1, 5, 10, 15],
            "model__min_child_samples": [10, 20, 30, 50],
            "model__subsample": [0.7, 0.85, 1.0],
            "model__colsample_bytree": [0.7, 0.85, 1.0],
            "model__class_weight": [None, "balanced"]
        }
    }
}

## Define search strategy

Two hyperparameter-search strategies will be used according to the size of each model's search space.

### Gaussian Naive Bayes: GridSearchCV

`GridSearchCV` will be used for `GaussianNB` because this model has only one hyperparameter in the current search space: `var_smoothing`.

The search contains five possible values, so testing every configuration is computationally inexpensive. An exhaustive grid search ensures that all proposed values are evaluated using cross-validation.

### Remaining models: RandomizedSearchCV

`RandomizedSearchCV` will be used for Logistic Regression, Gradient Boosting, Support Vector Machine, and LightGBM because their search spaces contain many possible hyperparameter combinations.

Evaluating every combination with `GridSearchCV` would require a considerably larger number of model fits. Instead, randomized search evaluates a fixed number of randomly selected configurations, reducing execution time while still exploring different regions of each model's hyperparameter space.

The same cross-validation strategy and evaluation metrics will be applied to all models to ensure a consistent comparison. The search will prioritize the F1-score for class `1`, since the objective is to balance churn recall and precision.

### Optimization metric

F1-score for class 1 is used as the optimization metric.

Because the target is imbalanced and churn detection requires both
detecting positive cases and controlling false positives, accuracy
alone is not considered sufficient for hyperparameter selection.

## Hyperparameter Search and Model Refitting

A single tuning loop dynamically selects GridSearchCV or
RandomizedSearchCV according to each model's configuration.

Each search object:
1. evaluates hyperparameter configurations using cross-validation;
2. ranks configurations according to F1;
3. identifies the best hyperparameters;
4. refits the winning Pipeline using the complete training set.

In [11]:
results = {}

for model_name, pipeline in models_pipeline.items():

    config = parameter_grids[model_name]

    if config["search_type"] == "grid":
        search = GridSearchCV(
            estimator=pipeline,
            param_grid=config["params"],
            scoring="f1",
            cv=3,
            n_jobs=-1
        )

    else:
        search = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=config["params"],
            n_iter=10,
            scoring="f1",
            cv=3,
            n_jobs=-1,
            random_state=42
        )

    search.fit(X_train, y_train)

    results[model_name] = search

/Users/emiliogarcialopez/projectGitHub/telco-churn-project/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/emiliogarcialopez/projectGitHub/telco-churn-project/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/emiliogarcialopez/projectGitHub/telco-churn-project/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/emiliogarcialopez/projectGitHub/telco-churn-project/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/emiliogarcialopez/projectGitHub/telco-churn-project/.venv/lib/python3.12/site-packages/sklearn/svm/_b

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

## Inspect Best Hyperparameters

The best hyperparameter configuration and corresponding mean CV
F1-score are extracted from each search object.

These results summarize the strongest configuration discovered for
each candidate before comparing model stability.

In [12]:
best_paragrams = {
    model_name: {
        "best_param": search.best_params_,
        "best_score": search.best_score_,
        "n_splits": search.n_splits_
    }
    for model_name, search in results.items()
}
display(best_paragrams)

{'logistic_regression': {'best_param': {'model__solver': 'saga',
   'model__dual': False,
   'model__class_weight': 'balanced',
   'model__C': 0.5},
  'best_score': np.float64(0.6326662691661392),
  'n_splits': 3},
 'gradient_boosting': {'best_param': {'model__subsample': 0.7,
   'model__n_estimators': 50,
   'model__min_samples_split': 2,
   'model__min_samples_leaf': 4,
   'model__max_depth': 2,
   'model__learning_rate': 0.2},
  'best_score': np.float64(0.5954034795045792),
  'n_splits': 3},
 'support_vector_machine': {'best_param': {'model__kernel': 'rbf',
   'model__gamma': 'scale',
   'model__class_weight': 'balanced',
   'model__C': 1},
  'best_score': np.float64(0.6223391333852537),
  'n_splits': 3},
 'gaussian_naive_bayes': {'best_param': {'model__var_smoothing': 1e-07},
  'best_score': np.float64(0.59648850854488),
  'n_splits': 3},
 'lightgbm': {'best_param': {'model__subsample': 1.0,
   'model__num_leaves': 63,
   'model__n_estimators': 300,
   'model__min_child_samples': 2

In [13]:
best_models = {
    model_name: search.best_estimator_
    for model_name, search in results.items()
}

## Inspect Cross-Validation Results

mean_test_score
→ average validation F1 across CV folds

std_test_score
→ variability of the F1 score across folds

rank_test_score
→ ranking of hyperparameter configurations within the same model

In [14]:
cv_dfs = {}

for model_name, search in results.items():
    df = pd.DataFrame(search.cv_results_)

    cv_dfs[model_name] = (
        df[
            [
                "mean_test_score",
                "std_test_score",
                "rank_test_score"
            ]
        ]
        .sort_values("rank_test_score")
        .reset_index(drop=True)
    )

In [15]:
for i in cv_dfs:
    display(i)
    display(cv_dfs[i])

'logistic_regression'

,mean_test_score,std_test_score,rank_test_score
0,0.632666,0.026693,1
1,0.632514,0.026922,2
2,0.628397,0.024375,3
3,0.597578,0.018042,4
4,0.597481,0.019844,5
5,0.583609,0.016528,6
6,NaN,NaN,7
7,NaN,NaN,7
8,NaN,NaN,7
9,NaN,NaN,7


'gradient_boosting'

,mean_test_score,std_test_score,rank_test_score
0,0.595403,0.009317,1
1,0.587310,0.017822,2
2,0.584785,0.016929,3
3,0.582197,0.014978,4
4,0.580913,0.018363,5
5,0.578773,0.010873,6
6,0.578637,0.010848,7
7,0.557991,0.016590,8
8,0.555707,0.022642,9
9,0.000000,0.000000,10


'support_vector_machine'

,mean_test_score,std_test_score,rank_test_score
0,0.622339,0.020934,1
1,0.621809,0.020591,2
2,0.615347,0.017992,3
3,0.615252,0.016775,4
4,0.614041,0.021841,5
5,0.611379,0.024774,6
6,0.595303,0.018178,7
7,0.586361,0.019530,8
8,0.579443,0.010890,9
9,0.568990,0.014782,10


'gaussian_naive_bayes'

,mean_test_score,std_test_score,rank_test_score
0,0.596489,0.018012,1
1,0.596489,0.018012,1
2,0.596489,0.018012,1
3,0.596489,0.018012,1
4,0.596489,0.018012,1


'lightgbm'

,mean_test_score,std_test_score,rank_test_score
0,0.633663,0.018237,1
1,0.633435,0.016814,2
2,0.633018,0.020160,3
3,0.632194,0.015385,4
4,0.631299,0.020176,5
5,0.630520,0.018903,6
6,0.630491,0.018383,7
7,0.567638,0.014496,8
8,0.565983,0.028185,9
9,0.553112,0.017481,10


In [16]:
tuning_summary = pd.DataFrame([
    {
        "model": model_name,
        "best_cv_f1": search.best_score_,
        "best_cv_std": pd.DataFrame(search.cv_results_)
            .loc[search.best_index_, "std_test_score"],
        "best_params": search.best_params_
    }
    for model_name, search in results.items()
])

### Inspect LightGBM Configurations

Several LightGBM configurations produced similar CV F1 scores,
suggesting that the candidate's performance is not dependent on a
single isolated hyperparameter combination.

The highest-ranked configuration achieves the strongest mean F1 while
maintaining acceptable fold-to-fold variability.

In [17]:
cv_results = pd.DataFrame(results["lightgbm"].cv_results_)

cv_results.sort_values("rank_test_score")

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__subsample,param_model__num_leaves,param_model__n_estimators,param_model__min_child_samples,param_model__max_depth,param_model__learning_rate,param_model__colsample_bytree,param_model__class_weight,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score
2,7.938952,0.117784,0.031344,0.008833,1.00,63,300,20,5,0.01,0.70,balanced,"{'model__subsample': 1.0, 'model__num_leaves':...",0.644152,0.648824,0.608013,0.633663,0.018237,1
9,1.856448,0.022215,0.021346,0.001025,0.70,63,100,50,5,0.10,0.85,balanced,"{'model__subsample': 0.7, 'model__num_leaves':...",0.643392,0.647157,0.609756,0.633435,0.016814,2
5,2.038646,0.269425,0.007882,0.000661,0.85,15,100,30,15,0.05,0.70,balanced,"{'model__subsample': 0.85, 'model__num_leaves'...",0.650206,0.644122,0.604724,0.633018,0.020160,3
3,7.592457,0.105343,0.064373,0.038934,0.85,63,100,10,-1,0.01,0.70,balanced,"{'model__subsample': 0.85, 'model__num_leaves'...",0.643463,0.642678,0.610442,0.632194,0.015385,4
0,32.893664,0.706913,0.028226,0.002752,0.85,63,500,20,10,0.01,0.85,balanced,"{'model__subsample': 0.85, 'model__num_leaves'...",0.641048,0.649648,0.603201,0.631299,0.020176,5
8,18.647415,6.185830,0.016215,0.001793,0.85,63,500,30,10,0.01,0.70,balanced,"{'model__subsample': 0.85, 'model__num_leaves'...",0.643042,0.644714,0.603805,0.630520,0.018903,6
4,3.697919,0.048184,0.012744,0.003534,0.85,15,200,50,15,0.01,0.70,balanced,"{'model__subsample': 0.85, 'model__num_leaves'...",0.638436,0.647955,0.605081,0.630491,0.018383,7
1,7.919453,0.099641,0.016219,0.001563,1.00,31,500,50,5,0.05,0.70,NaN,"{'model__subsample': 1.0, 'model__num_leaves':...",0.587845,0.554545,0.560523,0.567638,0.014496,8
6,32.225048,0.243812,0.018128,0.002358,0.85,63,500,10,15,0.01,0.70,NaN,"{'model__subsample': 0.85, 'model__num_leaves'...",0.605263,0.552204,0.540481,0.565983,0.028185,9
7,23.459450,0.339642,0.069174,0.040885,0.85,63,500,30,10,0.05,1.00,NaN,"{'model__subsample': 0.85, 'model__num_leaves'...",0.577586,0.537853,0.543897,0.553112,0.017481,10


## Save Tuning Results

The tuning stage produced two outputs that must be preserved for the next notebooks:

- A summary of the cross-validation performance of all tuned candidate models.
- The best hyperparameter configuration found for the selected LightGBM candidate.

These artifacts allow the next stages to reuse the tuning results without repeating the hyperparameter search.

The selected model is not yet considered the final production model. Threshold selection, business-cost analysis, interpretability, and final evaluation on the untouched test set are still pending.

In [18]:
selected_model = results["lightgbm"].best_estimator_

selected_params = results["lightgbm"].best_params_
selected_cv_score = results["lightgbm"].best_score_

In [19]:
tuning_summary.to_csv(
    "../data/processed/tuning_summary.csv",
    index=False
)

## Selected Candidate Configuration

LightGBM was selected as the leading candidate based exclusively on cross-validation results from the training data.

The selection considered both predictive performance and stability across folds. LightGBM achieved the highest mean F1-score among the tuned candidates while maintaining relatively low variability.

The best estimator returned by the hyperparameter search contains the complete fitted pipeline, including preprocessing and the optimized LightGBM classifier.

The selected hyperparameters and cross-validation score are extracted here so they can be documented and reused in subsequent stages.

In [20]:
joblib.dump(
    selected_model,
    "../artifacts/selected_lightgbm_pipeline.joblib"
)

['../artifacts/selected_lightgbm_pipeline.joblib']

## Tuning Outcome

The hyperparameter-tuning stage is now complete.

At this point, the project has identified the strongest candidate model and its best discovered hyperparameter configuration. However, the decision boundary has not yet been optimized.

The next stage will use validation probabilities from the selected LightGBM pipeline to analyze the precision-recall trade-off and determine an operating threshold according to business costs.

## Conclusion

- Five candidate algorithms were tuned.
- Gaussian Naive Bayes used exhaustive GridSearchCV because its search space was small.
- Logistic Regression, Gradient Boosting, Support Vector Machine, and LightGBM used RandomizedSearchCV to reduce computational cost while exploring larger search spaces.
- F1-score for the churn class was used as the optimization objective.
- Candidate selection was based exclusively on cross-validation results from the training data.
- Both mean validation performance and variability across folds were considered.
- LightGBM achieved the strongest combination of mean CV F1-score and stability.
- LightGBM was therefore selected as the leading candidate for the threshold and business-analysis stage.
- The held-out test set remains reserved for final evaluation.